In [2]:
from loguru import logger
from pathlib import Path
from typing import Dict, List, Union, Set
import json
import pandas as pd

In [3]:
ROOT_PATH = Path.cwd().parent
RAW_DATA_PATH = ROOT_PATH / "data" / "raw"
PROCESSED_DATA_PATH = ROOT_PATH / "data" / "processed"

FILENAME = "feature_dictionary.json"
OUTPUT_FILE = PROCESSED_DATA_PATH / FILENAME

In [4]:
class ComprehensiveFeatureAuditor:
    """
    Audits datasets by scanning every single record to identify all possible
    features, utilizing memory-efficient parsing for dynamic JSON schemas.
    """

    def __init__(self, raw_data_dir: Union[str, Path]):
        self.raw_data_dir = Path(raw_data_dir)
        self.feature_dictionary: Dict[str, List[str]] = {}

    def _extract_csv_features(self, filepath: Path) -> List[str]:
        """Scans CSV header to extract column names.

        Fetch nrows=0 is slightly more efficient than nrows=1 as it fetches only the schema
        """
        try:
            df = pd.read_csv(filepath, nrows=0)
            return df.columns.tolist()
        except Exception as e:
            logger.error(f"Error auditing CSV {filepath.name}: {e}")
            return []

    def _extract_json_features(self, filepath: Path) -> List[str]:
        """
        Memory-efficiently iterates through every record in a JSON array to find
        the union of all possible nested keys without Pandas initialization overhead.
        """
        all_keys: Set[str] = set()
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if not isinstance(data, list):
                logger.warning(f"Expected a list of objects in {filepath.name}, got {type(data)}")
                return []

            logger.info(f"Scanning {len(data)} records in {filepath.name} using pure Python traversal")

            def extract_keys(obj: dict, prefix: str = ''):
                """
                Subroutine to recursively extract keys from nested dictionaries, ensure keys are
                flattened with dot notation to match pd.json_normalize output and extract all
                possible keys across all records.
                """
                if isinstance(obj, dict):
                    for k, v in obj.items():
                        new_key = f"{prefix}.{k}" if prefix else k
                        if isinstance(v, dict):
                            extract_keys(v, new_key)
                        else:
                            all_keys.add(new_key)

            for record in data:
                extract_keys(record)

            return sorted(list(all_keys))

        except Exception as e:
            logger.error(f"Error auditing JSON {filepath.name}: {e}")
            return []

    def run_audit(self) -> Dict[str, List[str]]:
        """Orchestrates the feature audit across all files in the raw data directory."""
        if not self.raw_data_dir.exists():
            logger.error(f"Directory not found: {self.raw_data_dir}")
            return {}

        for filepath in self.raw_data_dir.rglob("*"):
            if filepath.suffix.lower() == '.csv':
                logger.info(f"Auditing CSV: {filepath.name}")
                self.feature_dictionary[filepath.name] = self._extract_csv_features(filepath)

            elif filepath.suffix.lower() == '.json':
                logger.info(f"Auditing Dynamic JSON: {filepath.name}")
                self.feature_dictionary[filepath.name] = self._extract_json_features(filepath)

        return self.feature_dictionary

    def export_report(self, output_path: Union[str, Path]) -> None:
        """Saves the union of all discovered features to a JSON report."""
        output_file = Path(output_path)
        output_file.parent.mkdir(parents=True, exist_ok=True)

        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(self.feature_dictionary, f, indent=4, ensure_ascii=False)
            logger.info(f"Comprehensive report exported to: {output_file}")
        except Exception as e:
            logger.error(f"Failed to export report: {e}")

In [5]:
auditor = ComprehensiveFeatureAuditor(raw_data_dir=RAW_DATA_PATH)
auditor.run_audit()
auditor.export_report(output_path=OUTPUT_FILE)

2026-03-12 14:16:41.697 | INFO     | __main__:run_audit:70 - Auditing CSV: data xe dien bonbanh.com.csv
2026-03-12 14:16:41.715 | INFO     | __main__:run_audit:70 - Auditing CSV: data_xe_dien_web_otodien.csv
2026-03-12 14:16:41.724 | INFO     | __main__:run_audit:70 - Auditing CSV: xevinfastluot_full.csv
2026-03-12 14:16:41.728 | INFO     | __main__:run_audit:74 - Auditing Dynamic JSON: cars.json
2026-03-12 14:16:41.931 | INFO     | __main__:_extract_json_features:37 - Scanning 1666 records in cars.json using pure Python traversal
2026-03-12 14:16:41.985 | INFO     | __main__:export_report:87 - Comprehensive report exported to: C:\Users\dduya\Work\project\ev_car\data\processed\feature_dictionary.json


In [6]:
for k, i in auditor.feature_dictionary.items():
    print(f"{k}: {len(i)} features")

data xe dien bonbanh.com.csv: 19 features
data_xe_dien_web_otodien.csv: 34 features
xevinfastluot_full.csv: 19 features
cars.json: 186 features


In [8]:
auditor.feature_dictionary["data xe dien bonbanh.com.csv"]

['Ngày đăng',
 'Tên xe',
 'Giá',
 'Tên người bán',
 'Địa chỉ',
 'Năm sản xuất',
 'Tình trạng',
 'Số Km đã đi',
 'Xuất xứ',
 'Kiểu dáng',
 'Hộp số',
 'Động cơ',
 'Màu ngoại thất',
 'Màu nội thất',
 'Số chỗ ngồi',
 'Số cửa',
 'Dẫn động',
 'Mô tả',
 'Link']

In [9]:
auditor.feature_dictionary["xevinfastluot_full.csv"]

['Ngày đăng',
 'Tên xe',
 'Giá',
 'Tên người bán',
 'Địa chỉ',
 'Năm sản xuất',
 'Tình trạng',
 'Số Km đã đi',
 'Xuất xứ',
 'Kiểu dáng',
 'Hộp số',
 'Động cơ',
 'Màu ngoại thất',
 'Màu nội thất',
 'Số chỗ ngồi',
 'Số cửa',
 'Dẫn động',
 'Mô tả',
 'Link']